# Lenguajes y Autómatas II (SCD-1016)
## Sesión 31 · Introducción a Ensamblador x86-64 y Arquitectura de CPU

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se convierte una operación abstracta como t3 = t1 + t2 en instrucciones que la CPU entiende?
>
> **Objetivo de la sesión:** Traducir un cuádruplo TAC a la secuencia x86-64 que lo ejecuta y comprobar que el valor calculado coincide con la operación original.

**Entregable:** tu implementación de mnemonico() y ejecutar_tac() en artesano_sesion31_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion31.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un operador (+, -, *) o un cuádruplo con la forma tN = A op B, y devuelves el mnemónico AT&T correspondiente o el valor que la CPU obtendría al ejecutarlo.

Al final de la sesión tendrás: **Dos funciones en C: una que nombra la instrucción x86-64 de cada operador TAC y otra que ejecuta el cuádruplo con la semántica de la CPU.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 31](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion31.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion31.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion31.h"

const char *mnemonico(char op);
long ejecutar_tac(const char *tac);
```

- Los operandos del cuádruplo son literales decimales con signo; no hay variables ni acceso a memoria.
- Un operador desconocido se responde con nop, y un cuádruplo mal formado vale cero.
- Nada de main(): el harness ya trae el suyo y compila con -Werror.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion31_c.c` junto con `artesano_sesion31_c.c`
> (o `ia_sesion31_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `mnemonico('+')` | `addq` |
| 2 | `mnemonico('*')` | `imulq` |
| 3 | `mnemonico('?')` | `nop` |
| 4 | `ejecutar_tac("t0 = 4 + 5")` | 9 |
| 5 | `ejecutar_tac("t1 = 10 - 25")` | -15 |
| 6 | `ejecutar_tac("t2 = 7 * 6")` | 42 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 31 - Mapeo de TAC a instrucciones mov/add/sub ===
Caso                               esperado:              obtenido:              resultado
[ 1] mnemonico('+')                     esperado: addq         obtenido: addq         OK
[ 2] mnemonico('*')                     esperado: imulq        obtenido: imulq        OK
[ 3] mnemonico('?')                     esperado: nop          obtenido: nop          OK
[ 4] ejecutar_tac("t0 = 4 + 5")         esperado: 9            obtenido: 9            OK
[ 5] ejecutar_tac("t1 = 10 - 25")       esperado: -15          obtenido: -15          OK
[ 6] ejecutar_tac("t2 = 7 * 6")         esperado: 42           obtenido: 42           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion31.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] mnemonico devuelve exactamente el mnemónico AT&T de cada operador, sin prefijos ni sufijos extra.
- [ ] ejecutar_tac aplica la operación del cuádruplo y devuelve el mismo valor que produciría la CPU.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion31_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion31.h
/* api_sesion31.h - contrato de la Sesion 31
 * Mapeo de TAC a instrucciones mov/add/sub
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION31_H
#define API_SESION31_H

const char *mnemonico(char op);
long ejecutar_tac(const char *tac);

#endif /* API_SESION31_H */


In [ ]:
%%writefile artesano_sesion31_c.c
/* artesano_sesion31_c.c - Fase 1: El Artesano (sin IA)
 * Mapeo de TAC a instrucciones mov/add/sub.
 *
 * Implementa lo que declara api_sesion31.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion31.h"
#include <stdio.h>
#include <string.h>

const char *mnemonico(char op) {
    (void)op; /* TODO: + -> addq, - -> subq, * -> imulq, otro -> nop */
    return "nop";
}

long ejecutar_tac(const char *tac) {
    (void)tac; /* TODO: leer tN = A op B y aplicar la operacion */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion31_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion31_c.c
/* ia_sesion31_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion31.h. No puede traer main().
 */
#include "api_sesion31.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion31_c.c
/* harness_sesion31_c.c - auditoria automatica de la Sesion 31
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion31_c.c artesano_sesion31_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion31.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 31 - Mapeo de TAC a instrucciones mov/add/sub ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { const char *r = mnemonico('+'); verificar_str("mnemonico('+')", "addq", r); }
    { const char *r = mnemonico('*'); verificar_str("mnemonico('*')", "imulq", r); }
    { const char *r = mnemonico('?'); verificar_str("mnemonico('?')", "nop", r); }
    { long r = ejecutar_tac("t0 = 4 + 5"); verificar_int("ejecutar_tac(\"t0 = 4 + 5\")", (long)(9), (long)(r)); }
    { long r = ejecutar_tac("t1 = 10 - 25"); verificar_int("ejecutar_tac(\"t1 = 10 - 25\")", (long)(-15), (long)(r)); }
    { long r = ejecutar_tac("t2 = 7 * 6"); verificar_int("ejecutar_tac(\"t2 = 7 * 6\")", (long)(42), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion31.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion31_c.c artesano_sesion31_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion31_c.c ia_sesion31_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion31.md` a `auditorias/auditoria_sesion31.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
